In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('final_enriched_dataset (1).csv')

# Quick sanity check
print(df.head())
print(df.info())

       Country        date  agricultural_land%  forest_land%  land_area  \
0  AFGHANISTAN  1960-01-01                 NaN           NaN        NaN   
1  AFGHANISTAN  1961-01-01           57.878356           NaN   652230.0   
2  AFGHANISTAN  1962-01-01           57.955016           NaN   652230.0   
3  AFGHANISTAN  1963-01-01           58.031676           NaN   652230.0   
4  AFGHANISTAN  1964-01-01           58.116002           NaN   652230.0   

   avg_precipitation  trade_in_services%  control_of_corruption_estimate  \
0                NaN                 NaN                             NaN   
1              327.0                 NaN                             NaN   
2              327.0                 NaN                             NaN   
3              327.0                 NaN                             NaN   
4              327.0                 NaN                             NaN   

   control_of_corruption_std  access_to_electricity%  ...  \
0                        NaN   

In [ ]:
print(df.isnull().sum())

Country                                                                               0
date                                                                                  0
agricultural_land%                                                                  690
forest_land%                                                                       2803
land_area                                                                           680
                                                                                   ... 
People using at least basic sanitation services (% of population)                  3505
School enrollment, primary (% net)                                                 3495
School enrollment, secondary (% net)                                               4105
School enrollment, tertiary (gross), gender parity index (GPI)                     3380
Unemployment with basic education (% of total labor force with basic education)    4440
Length: 64, dtype: int64


In [ ]:
structural_constraints = ['avg_precipitation','land_area']
df = df.sort_values(['Country', 'date'])

df[structural_constraints] = (
    df.groupby('Country')[structural_constraints]
      .transform(lambda group: group.ffill().bfill())
)

In [ ]:
df.head()

,Country,date,agricultural_land%,forest_land%,land_area,avg_precipitation,trade_in_services%,control_of_corruption_estimate,control_of_corruption_std,access_to_electricity%,...,"Literacy rate, adult total (% of people ages 15 and above)","Literacy rate, youth (ages 15-24), gender parity index (GPI)","Mortality rate, infant (per 1,000 live births)","Mortality rate, under-5 (per 1,000 live births)",People using at least basic drinking water services (% of population),People using at least basic sanitation services (% of population),"School enrollment, primary (% net)","School enrollment, secondary (% net)","School enrollment, tertiary (gross), gender parity index (GPI)",Unemployment with basic education (% of total labor force with basic education)
0,AFGHANISTAN,1960-01-01,NaN,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,251.2,353.2,NaN,NaN,NaN,NaN,NaN,NaN
1,AFGHANISTAN,1961-01-01,57.878356,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,248.4,347.6,NaN,NaN,NaN,NaN,NaN,NaN
2,AFGHANISTAN,1962-01-01,57.955016,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,245.4,342.3,NaN,NaN,NaN,NaN,NaN,NaN
3,AFGHANISTAN,1963-01-01,58.031676,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,242.5,336.8,NaN,NaN,NaN,NaN,NaN,NaN
4,AFGHANISTAN,1964-01-01,58.116002,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,239.7,331.7,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
slow_moving_environment_variables = ['agricultural_land%', 'forest_land%']

# Clean column names just in case
df.columns = df.columns.str.strip().str.lower()

# Sort properly
df = df.sort_values(['country', 'date'])

# Apply country-wise logic
df[slow_moving_environment_variables] = (
    df.groupby('country')[slow_moving_environment_variables]
      .transform(lambda x: x.interpolate(method='linear', limit=2).ffill())
)

In [ ]:
df.head()

,country,date,agricultural_land%,forest_land%,land_area,avg_precipitation,trade_in_services%,control_of_corruption_estimate,control_of_corruption_std,access_to_electricity%,...,"literacy rate, adult total (% of people ages 15 and above)","literacy rate, youth (ages 15-24), gender parity index (gpi)","mortality rate, infant (per 1,000 live births)","mortality rate, under-5 (per 1,000 live births)",people using at least basic drinking water services (% of population),people using at least basic sanitation services (% of population),"school enrollment, primary (% net)","school enrollment, secondary (% net)","school enrollment, tertiary (gross), gender parity index (gpi)",unemployment with basic education (% of total labor force with basic education)
0,AFGHANISTAN,1960-01-01,NaN,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,251.2,353.2,NaN,NaN,NaN,NaN,NaN,NaN
1,AFGHANISTAN,1961-01-01,57.878356,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,248.4,347.6,NaN,NaN,NaN,NaN,NaN,NaN
2,AFGHANISTAN,1962-01-01,57.955016,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,245.4,342.3,NaN,NaN,NaN,NaN,NaN,NaN
3,AFGHANISTAN,1963-01-01,58.031676,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,242.5,336.8,NaN,NaN,NaN,NaN,NaN,NaN
4,AFGHANISTAN,1964-01-01,58.116002,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,239.7,331.7,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
monotonic_development_variables = ['access_to_electricity%', 'individuals_using_internet%', 'people using at least basic drinking water services (% of population)', 'people using at least basic sanitation services (% of population)']

df.columns = df.columns.str.strip().str.lower()
df = df.sort_values(['country', 'date'])

def process_monotonic(group):
    group = group.interpolate(method='linear', limit=2)
    group = group.cummax()
    group = group.ffill()
    return group

df[monotonic_development_variables] = (
    df.groupby('country')[monotonic_development_variables]
      .transform(process_monotonic)
)

In [ ]:
df.head()

,country,date,agricultural_land%,forest_land%,land_area,avg_precipitation,trade_in_services%,control_of_corruption_estimate,control_of_corruption_std,access_to_electricity%,...,"literacy rate, adult total (% of people ages 15 and above)","literacy rate, youth (ages 15-24), gender parity index (gpi)","mortality rate, infant (per 1,000 live births)","mortality rate, under-5 (per 1,000 live births)",people using at least basic drinking water services (% of population),people using at least basic sanitation services (% of population),"school enrollment, primary (% net)","school enrollment, secondary (% net)","school enrollment, tertiary (gross), gender parity index (gpi)",unemployment with basic education (% of total labor force with basic education)
0,AFGHANISTAN,1960-01-01,NaN,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,251.2,353.2,NaN,NaN,NaN,NaN,NaN,NaN
1,AFGHANISTAN,1961-01-01,57.878356,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,248.4,347.6,NaN,NaN,NaN,NaN,NaN,NaN
2,AFGHANISTAN,1962-01-01,57.955016,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,245.4,342.3,NaN,NaN,NaN,NaN,NaN,NaN
3,AFGHANISTAN,1963-01-01,58.031676,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,242.5,336.8,NaN,NaN,NaN,NaN,NaN,NaN
4,AFGHANISTAN,1964-01-01,58.116002,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,239.7,331.7,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
structural_break_variables = ['co2_emisions','other_greenhouse_emisions','renewvable_energy_consumption%','goverment_effectiveness_estimate','goverment_effectiveness_std','control_of_corruption_estimate','control_of_corruption_std','political_stability_estimate','political_stability_std','rule_of_law_std','regulatory_quality_estimate','regulatory_quality_std','voice_and_accountability_estimate','voice_and_accountability_std']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df = df.sort_values(['country', 'date'])

def process_structural(group):
    # Step 1: Interpolate small gaps (≤2)
    interp = group.interpolate(method='linear', limit=2)

    # Step 2: Forward fill medium gaps (≤5 total after interpolation)
    filled = interp.ffill(limit=3)

    # Step 3: Preserve pre-start NaNs
    first_valid_idx = group.first_valid_index()
    if first_valid_idx is not None:
        filled.loc[:first_valid_idx] = group.loc[:first_valid_idx]

    return filled

df[structural_break_variables] = (
    df.groupby('country')[structural_break_variables]
      .transform(process_structural)
)

In [ ]:
df.head()

,country,date,agricultural_land%,forest_land%,land_area,avg_precipitation,trade_in_services%,control_of_corruption_estimate,control_of_corruption_std,access_to_electricity%,...,"literacy rate, adult total (% of people ages 15 and above)","literacy rate, youth (ages 15-24), gender parity index (gpi)","mortality rate, infant (per 1,000 live births)","mortality rate, under-5 (per 1,000 live births)",people using at least basic drinking water services (% of population),people using at least basic sanitation services (% of population),"school enrollment, primary (% net)","school enrollment, secondary (% net)","school enrollment, tertiary (gross), gender parity index (gpi)",unemployment with basic education (% of total labor force with basic education)
0,AFGHANISTAN,1960-01-01,NaN,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,251.2,353.2,NaN,NaN,NaN,NaN,NaN,NaN
1,AFGHANISTAN,1961-01-01,57.878356,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,248.4,347.6,NaN,NaN,NaN,NaN,NaN,NaN
2,AFGHANISTAN,1962-01-01,57.955016,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,245.4,342.3,NaN,NaN,NaN,NaN,NaN,NaN
3,AFGHANISTAN,1963-01-01,58.031676,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,242.5,336.8,NaN,NaN,NaN,NaN,NaN,NaN
4,AFGHANISTAN,1964-01-01,58.116002,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,239.7,331.7,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
moderately_smooth_economic_series = ['tax_revenue%','expense%','military_expenditure%','government_expenditure_on_education%','government_health_expenditure%']

In [ ]:

df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values(['country', 'date'])

def process_moderate(group):
    original = group.copy()

    # Step 1: Interpolate small + medium gaps (≤5)
    group = group.interpolate(method='linear', limit=5)

    # Step 2: Sanity check (prevent unrealistic jumps)
    pct_change = group.pct_change(fill_method = None)
    group[pct_change.abs() > 0.5] = original  # revert if >50% jump

    # Step 3: Forward fill only near end (limit=2)
    group = group.ffill(limit=2)

    # Step 4: Preserve pre-start NaNs
    first_valid = original.first_valid_index()
    if first_valid is not None:
        group.loc[:first_valid] = original.loc[:first_valid]

    return group

df[moderately_smooth_economic_series] = (
    df.groupby('country')[moderately_smooth_economic_series]
      .transform(process_moderate)
)

In [ ]:
df[df['country'] == 'INDIA'][moderately_smooth_economic_series].tail(20)

,tax_revenue%,expense%,military_expenditure%,government_expenditure_on_education%,government_health_expenditure%


In [ ]:
high_growth_macro_series = ['gdp_current_us']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

def log_interpolate(group):
    # Step 1: replace 0 with NaN
    group = group.replace(0, np.nan)

    # Step 2: take log
    log_values = np.log(group)

    # Step 3: interpolate in log space
    log_values = log_values.interpolate(method='linear')

    # Step 4: convert back
    return np.exp(log_values)

df[high_growth_macro_series] = (
    df.groupby('country')[high_growth_macro_series]
      .transform(log_interpolate)
)

In [ ]:
df[df['country'] == 'AFGHANISTAN'][high_growth_macro_series].tail(20)

,gdp_current_us
44,5.224897e+09
45,6.203256e+09
46,6.971758e+09
47,9.747886e+09
48,1.010930e+10
49,1.241615e+10
50,1.585667e+10
51,1.780510e+10
52,1.990733e+10
53,2.014642e+10


In [ ]:
dynamic_economic_series = ['exports of goods and services (% of gdp)',
    'imports of goods and services (% of gdp)',
    'electric_power_consumption',
    'trade_in_services%']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

def process_dynamic(group):
    original = group.copy()

    # Step 1: interpolate only small gaps (≤2)
    group = group.interpolate(method='linear', limit=2)

    # Step 2: detect large gaps using original
    is_na = original.isna()
    gap_id = (is_na != is_na.shift()).cumsum()
    gap_size = is_na.groupby(gap_id).transform('sum')

    large_gap_mask = (is_na) & (gap_size > 2)

    # Step 3: apply growth-based fill ONLY for small extension (limit=1)
    growth = group.pct_change(fill_method=None)
    growth_filled = group.ffill(limit=1) * (1 + growth.fillna(0))

    # Step 4: keep large gaps mostly NaN (don’t overfill)
    group[large_gap_mask] = growth_filled[large_gap_mask]

    return group

df[dynamic_economic_series] = (
    df.groupby('country')[dynamic_economic_series]
      .transform(process_dynamic)
)

In [ ]:
df[df['country'] == 'CHINA'][dynamic_economic_series].tail(20)

,exports of goods and services (% of gdp),imports of goods and services (% of gdp),electric_power_consumption,trade_in_services%
2988,30.609716,28.030618,1585.838782,7.775606
2989,33.368798,27.991277,1782.312153,7.105957
2990,35.526847,28.042733,2039.014660,7.082150
2991,34.906696,26.359768,2325.926769,7.476764
2992,32.092514,24.618619,2446.369055,7.014336
2993,24.330772,20.088993,2612.456620,5.929275
2994,26.722745,23.131327,2943.589954,6.106928
2995,26.151883,23.793944,3295.784868,5.944359
2996,25.076704,22.403509,3466.019539,5.659465
2997,24.163343,21.752698,3757.185088,5.617418


In [ ]:
print(df.columns.tolist())

['country', 'date', 'agricultural_land%', 'forest_land%', 'land_area', 'avg_precipitation', 'trade_in_services%', 'control_of_corruption_estimate', 'control_of_corruption_std', 'access_to_electricity%', 'renewvable_energy_consumption%', 'electric_power_consumption', 'co2_emisions', 'other_greenhouse_emisions', 'population_density', 'inflation_annual%', 'real_interest_rate', 'risk_premium_on_lending', 'research_and_development_expenditure%', 'central_goverment_debt%', 'tax_revenue%', 'expense%', 'goverment_effectiveness_estimate', 'goverment_effectiveness_std', 'human_capital_index', 'doing_business', 'time_to_get_operation_license', 'statistical_performance_indicators', 'individuals_using_internet%', 'logistic_performance_index', 'military_expenditure%', 'gdp_current_us', 'political_stability_estimate', 'political_stability_std', 'rule_of_law_estimate', 'rule_of_law_std', 'regulatory_quality_estimate', 'regulatory_quality_std', 'government_expenditure_on_education%', 'government_health

In [ ]:
social_devlopemental_behaviour = ['labor force participation rate for ages 15-24, total (%) (national estimate)', 'literacy rate, adult total (% of people ages 15 and above)', 'literacy rate, youth (ages 15-24), gender parity index (gpi)','school enrollment, primary (% net)', 'school enrollment, secondary (% net)', 'school enrollment, tertiary (gross), gender parity index (gpi)']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

def process_social(group):
    original = group.copy()

    # Step 1: find first valid index
    first_valid = original.first_valid_index()

    if first_valid is None:
        return group  # all NaN, leave as is

    # Step 2: apply only AFTER first valid value
    after_start = group.loc[first_valid:].copy()

    # Interpolation (even large gaps)
    after_start = after_start.interpolate(method='linear')

    # Forward fill only small edge gaps (limit=2)
    after_start = after_start.ffill(limit=2)

    # Step 3: combine back
    group.loc[first_valid:] = after_start

    # Step 4: ensure pre-start remains NaN
    group.loc[:first_valid] = original.loc[:first_valid]

    return group

df[social_devlopemental_behaviour] = (
    df.groupby('country')[social_devlopemental_behaviour]
      .transform(process_social)
)

In [ ]:
df[df['country'] == 'CHINA'][social_devlopemental_behaviour].tail(40)

,"labor force participation rate for ages 15-24, total (%) (national estimate)","literacy rate, adult total (% of people ages 15 and above)","literacy rate, youth (ages 15-24), gender parity index (gpi)","school enrollment, primary (% net)","school enrollment, secondary (% net)","school enrollment, tertiary (gross), gender parity index (gpi)"
2968,81.70250,68.580002,0.880,NaN,NaN,0.387031
2969,81.13375,70.115002,0.890,NaN,NaN,0.400844
2970,80.56500,71.650002,0.900,NaN,NaN,0.414657
2971,79.99625,73.185001,0.910,92.84520,NaN,0.428470
2972,79.42750,74.720001,0.920,94.58020,NaN,0.442283
2973,78.85875,76.255001,0.930,96.13547,NaN,0.456096
2974,78.29000,77.790001,0.940,96.87553,NaN,0.469909
2975,77.04120,79.103001,0.945,96.82433,NaN,0.483721
2976,75.79240,80.416000,0.950,95.86940,NaN,0.497534
2977,74.54360,81.729000,0.955,94.73328,NaN,0.511347


In [ ]:
highly_volatile_variables = ['inflation_annual%','real_interest_rate','unemployment with basic education (% of total labor force with basic education)']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

def process_volatile(group):
    original = group.copy()

    is_na = original.isna()
    gap_id = (is_na != is_na.shift()).cumsum()
    gap_size = is_na.groupby(gap_id).transform('sum')

    small_gap = is_na & (gap_size <= 2)
    medium_gap = is_na & (gap_size.between(3, 5))
    large_gap = is_na & (gap_size > 5)

    # Small → interpolation
    interp = group.interpolate(method='linear', limit=2)

    # Medium → rolling median
    rolling_med = group.rolling(window=3, min_periods=1, center=True).median()

    # Fallback interpolation for medium gaps
    interp_medium = group.interpolate(method='linear', limit=5)

    result = group.copy()

    result[small_gap] = interp[small_gap]

    # Medium: first try rolling, then fallback
    result[medium_gap] = rolling_med[medium_gap]
    result[medium_gap & result.isna()] = interp_medium[medium_gap & result.isna()]

    # Large → keep NaN
    result[large_gap] = np.nan

    return result

df[highly_volatile_variables] = (
    df.groupby('country')[highly_volatile_variables]
      .transform(process_volatile)
)

In [ ]:
df[df['country'] == 'INDIA'][highly_volatile_variables].tail(40)

,inflation_annual%,real_interest_rate,unemployment with basic education (% of total labor force with basic education)


In [ ]:
structural_mnar_variables = ['research_and_development_expenditure%','central_goverment_debt%','gini_index','multidimensional_poverty_headcount_ratio%']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

def process_structural_mnar(group):
    original = group.copy()

    # Identify first valid point
    first_valid = original.first_valid_index()
    if first_valid is None:
        return group

    # Work only after first valid
    working = group.loc[first_valid:].copy()

    # Detect gaps
    is_na = working.isna()
    gap_id = (is_na != is_na.shift()).cumsum()
    gap_size = is_na.groupby(gap_id).transform('sum')

    small_gap = is_na & (gap_size <= 3)
    medium_gap = is_na & (gap_size.between(5, 10))

    # Step 1: interpolation for medium gaps
    interp = working.interpolate(method='linear', limit=10)

    # Step 2: forward fill for small gaps
    ffilled = working.ffill(limit=3)

    result = working.copy()

    result[medium_gap] = interp[medium_gap]
    result[small_gap] = ffilled[small_gap]

    # Step 3: tail handling (recent years)
    # identify last valid index
    last_valid = original.last_valid_index()
    if last_valid is not None:
        tail = result.loc[last_valid:]
        tail = tail.ffill().bfill()
        result.loc[last_valid:] = tail

    # Put back into original structure
    group.loc[first_valid:] = result

    # Ensure pre-start NaNs preserved
    group.loc[:first_valid] = original.loc[:first_valid]

    return group

df[structural_mnar_variables] = (
    df.groupby('country')[structural_mnar_variables]
      .transform(process_structural_mnar)
)

In [ ]:
df[df['country'] == 'CHINA'][structural_mnar_variables].tail(20)

,research_and_development_expenditure%,central_goverment_debt%,gini_index,multidimensional_poverty_headcount_ratio%
2988,1.21498,NaN,42.0,NaN
2989,1.30792,NaN,40.9,NaN
2990,1.36854,NaN,40.9,NaN
2991,1.37369,NaN,40.9,NaN
2992,1.44592,NaN,43.0,NaN
2993,1.66480,NaN,43.0,NaN
2994,1.71372,NaN,43.7,NaN
2995,1.78034,NaN,42.4,NaN
2996,1.91214,NaN,42.2,NaN
2997,1.99786,NaN,39.7,NaN


In [ ]:
df = df.drop(columns=['human_capital_index', 'statistical_performance_indicators','logistic_performance_index','year','doing_business','time_to_get_operation_license','risk_premium_on_lending','population_density'])

In [ ]:
demographic_smooth_variables = ['birth_rate','death_rate','life_expectancy_at_birth','population','rural_population','mortality rate, infant (per 1,000 live births)','mortality rate, under-5 (per 1,000 live births)']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

def process_demographic(group):
    original = group.copy()

    # Step 1: interpolate (fills internal gaps only)
    group = group.interpolate(method='linear')

    # Step 2: handle edges only
    # fill start gaps (bfill)
    group = group.bfill(limit=2)

    # fill end gaps (ffill)
    group = group.ffill(limit=2)

    return group

df[demographic_smooth_variables] = (
    df.groupby('country')[demographic_smooth_variables]
      .transform(process_demographic)
)

In [ ]:
df[df['country'] == 'RUSSIAN FEDERATION'][demographic_smooth_variables].tail(40)

,birth_rate,death_rate,life_expectancy_at_birth,population,rural_population,"mortality rate, infant (per 1,000 live births)","mortality rate, under-5 (per 1,000 live births)"


In [ ]:
new = ['intentional_homicides']

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date'])

# ---- availability flag (sparse data awareness) ----
df['homicide_available'] = (
    df.groupby('country')['intentional_homicides']
      .transform(lambda x: x.notna().sum() > 0)
      .astype(int)
)

def process_homicide(group):
    original = group.copy()

    # Identify NA runs and their sizes
    is_na = original.isna()
    gap_id = (is_na != is_na.shift()).cumsum()
    gap_size = is_na.groupby(gap_id).transform('sum')

    small_gap = is_na & (gap_size <= 2)
    medium_gap = is_na & (gap_size.between(3, 5))
    large_gap = is_na & (gap_size > 5)

    # --- Small gaps: careful interpolation (limited) ---
    interp_small = group.interpolate(method='linear', limit=2)

    # --- Medium gaps: rolling median (prefer centered) ---
    roll3 = group.rolling(window=3, center=True, min_periods=1).median()
    roll5 = group.rolling(window=5, center=True, min_periods=1).median()

    # Use 5 if available, else fallback to 3
    roll_med = roll5.where(~roll5.isna(), roll3)

    # If rolling still NaN (all neighbors NaN), fallback to limited interpolation
    interp_med = group.interpolate(method='linear', limit=5)

    # --- Construct result ---
    result = group.copy()

    # Small gaps
    result[small_gap] = interp_small[small_gap]

    # Medium gaps: rolling first, then fallback if still NaN
    med_mask = medium_gap
    result[med_mask] = roll_med[med_mask]
    fallback_mask = med_mask & result.isna()
    result[fallback_mask] = interp_med[fallback_mask]

    # Large gaps: keep NaN explicitly
    result[large_gap] = np.nan

    # IMPORTANT: do not alter non-NaN values → spikes remain untouched
    return result

df[new] = (
    df.groupby('country')[new]
      .transform(process_homicide)
)

In [ ]:
df[df['country'] == 'SYRIAN ARAB REPUBLIC'][new].tail(40)

,intentional_homicides


In [ ]:
df['population_density'] = df['population'].div(df['land_area'])
df['population_density'].replace([np.inf, -np.inf], np.nan, inplace=True)

/tmp/ipykernel_12131/2367195728.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['population_density'].replace([np.inf, -np.inf], np.nan, inplace=True)


In [ ]:
df['population_density'].head(40)

,population_density
0,13.219978
1,13.477056
2,13.751356
3,14.040239
4,14.343888
5,14.665298
6,14.999535
7,15.347393
8,15.711911
9,16.090166


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5377 entries, 0 to 5376
Data columns (total 58 columns):
 #   Column                                                                           Non-Null Count  Dtype         
---  ------                                                                           --------------  -----         
 0   country                                                                          5377 non-null   object        
 1   date                                                                             5377 non-null   datetime64[ns]
 2   agricultural_land%                                                               4849 non-null   float64       
 3   forest_land%                                                                     2739 non-null   float64       
 4   land_area                                                                        5284 non-null   float64       
 5   avg_precipitation                                                         

In [ ]:
df.to_csv('missing_treated.csv', index=False)

from google.colab import files
files.download('missing_treated.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
class1_cols = [
    'access_to_electricity%',
    'individuals_using_internet%',
    'renewvable_energy_consumption%',
    'government_expenditure_on_education%',
    'government_health_expenditure%',
    'school enrollment, primary (% net)',
    'school enrollment, secondary (% net)',
    'literacy rate, adult total (% of people ages 15 and above)',
    'literacy rate, youth (ages 15-24), gender parity index (gpi)',
    'people using at least basic drinking water services (% of population)',
    'people using at least basic sanitation services (% of population)'
]


In [ ]:
def min_max_normalize(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5  # neutral

    return (series - min_val) / (max_val - min_val)

df[class1_cols] = df.groupby('date')[class1_cols].transform(min_max_normalize)

In [ ]:
# List of Class 2 columns
class2_cols = [
    'co2_emisions',
    'other_greenhouse_emisions',
    'inflation_annual%',
    'birth_rate',
    'death_rate',
    'mortality rate, infant (per 1,000 live births)',
    'mortality rate, under-5 (per 1,000 live births)',
    'intentional_homicides',
    'multidimensional_poverty_headcount_ratio%',
    'unemployment with basic education (% of total labor force with basic education)'
]

# Reverse Min-Max normalization function
def reverse_min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5  # neutral

    return (max_val - series) / (max_val - min_val)

# Apply year-wise normalization
df[class2_cols] = df.groupby('date')[class2_cols].transform(reverse_min_max)

In [ ]:
# Class 3 columns
class3_cols = [
    'gdp_current_us',
    'population',
    'electric_power_consumption',
    'population_density'
]

# Step 1: Replace 0 with NaN (important)
df[class3_cols] = df[class3_cols].replace(0, np.nan)

# Step 2: Log transform
df_log = df[class3_cols].apply(lambda x: np.log(x))

# Step 3: Min-Max normalization (year-wise)
def min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (series - min_val) / (max_val - min_val)

df[class3_cols] = df_log.groupby(df['date']).transform(min_max)

In [ ]:
# Class 4 columns
class4_cols = [
    'control_of_corruption_estimate',
    'goverment_effectiveness_estimate',
    'political_stability_estimate',
    'rule_of_law_estimate',
    'regulatory_quality_estimate',
    'voice_and_accountability_estimate'
]

# Normalize from [-2.5, 2.5] → [0, 1]
df[class4_cols] = df[class4_cols].apply(lambda x: (x + 2.5) / 5)

In [ ]:
# Positive variables
class5_pos = [
    'trade_in_services%',
    'exports of goods and services (% of gdp)',
    'tax_revenue%',
    'research_and_development_expenditure%'
]

# Negative variables
class5_neg = [
    'imports of goods and services (% of gdp)',
    'central_goverment_debt%',
    'military_expenditure%',
    'gini_index',
    'expense%'
]

# Min-max function
def min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (series - min_val) / (max_val - min_val)

# Reverse min-max
def reverse_min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (max_val - series) / (max_val - min_val)

# Apply normalization (year-wise)
df[class5_pos] = df.groupby('date')[class5_pos].transform(min_max)
df[class5_neg] = df.groupby('date')[class5_neg].transform(reverse_min_max)

In [ ]:
# Class 6 columns
class6_cols = [
    'life_expectancy_at_birth',
    'labor force participation rate for ages 15-24, total (%) (national estimate)',
    'school enrollment, tertiary (gross), gender parity index (gpi)'
]

# Min-max normalization function
def min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (series - min_val) / (max_val - min_val)

# Apply year-wise normalization
df[class6_cols] = df.groupby('date')[class6_cols].transform(min_max)

In [ ]:
# Split handling
class7_pos = ['agricultural_land%', 'forest_land%']
class7_neg = ['rural_population']

df[class7_pos] = df.groupby('date')[class7_pos].transform(min_max)
df[class7_neg] = df.groupby('date')[class7_neg].transform(reverse_min_max)

In [ ]:
df.head()

,date,agricultural_land%,forest_land%,land_area,avg_precipitation,trade_in_services%,control_of_corruption_estimate,control_of_corruption_std,access_to_electricity%,renewvable_energy_consumption%,...,"mortality rate, infant (per 1,000 live births)","mortality rate, under-5 (per 1,000 live births)",people using at least basic drinking water services (% of population),people using at least basic sanitation services (% of population),"school enrollment, primary (% net)","school enrollment, secondary (% net)","school enrollment, tertiary (gross), gender parity index (gpi)",unemployment with basic education (% of total labor force with basic education),homicide_available,population_density
0,1960-01-01,0.679612,0.018894,652230.0,327.0,0.04016,0.544307,0.225828,0.595607,0.273095,...,0.084211,0.184751,0.316699,0.330746,0.047344,0.220069,0.132147,0.753956,1,0.527496
1,1961-01-01,0.618339,0.018894,652230.0,327.0,0.04016,0.544307,0.225828,0.595607,0.273095,...,0.086478,0.193319,0.316699,0.330746,0.047344,0.220069,0.132147,0.753956,1,0.526716
2,1962-01-01,0.623148,0.018894,652230.0,327.0,0.04016,0.544307,0.225828,0.595607,0.273095,...,0.089180,0.206034,0.316699,0.330746,0.047344,0.220069,0.132147,0.753956,1,0.526136
3,1963-01-01,0.628189,0.018894,652230.0,327.0,0.04016,0.544307,0.225828,0.595607,0.273095,...,0.089271,0.219148,0.316699,0.330746,0.047344,0.220069,0.132147,0.753956,1,0.526107
4,1964-01-01,0.631213,0.018894,652230.0,327.0,0.04016,0.544307,0.225828,0.595607,0.273095,...,0.086745,0.217023,0.316699,0.330746,0.047344,0.220069,0.132147,0.753956,1,0.526238


In [ ]:
df.to_csv('pre-processed dataset.csv')

from google.colab import files
files.download('pre-processed dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df.columns.tolist())

['date', 'agricultural_land%', 'forest_land%', 'land_area', 'avg_precipitation', 'trade_in_services%', 'control_of_corruption_estimate', 'control_of_corruption_std', 'access_to_electricity%', 'renewvable_energy_consumption%', 'electric_power_consumption', 'co2_emisions', 'other_greenhouse_emisions', 'inflation_annual%', 'real_interest_rate', 'research_and_development_expenditure%', 'central_goverment_debt%', 'tax_revenue%', 'expense%', 'goverment_effectiveness_estimate', 'goverment_effectiveness_std', 'individuals_using_internet%', 'military_expenditure%', 'gdp_current_us', 'political_stability_estimate', 'political_stability_std', 'rule_of_law_estimate', 'rule_of_law_std', 'regulatory_quality_estimate', 'regulatory_quality_std', 'government_expenditure_on_education%', 'government_health_expenditure%', 'multidimensional_poverty_headcount_ratio%', 'gini_index', 'birth_rate', 'death_rate', 'life_expectancy_at_birth', 'population', 'rural_population', 'voice_and_accountability_estimate', 

In [ ]:
print(df.index.names)

[None]


In [ ]:
info = ['country','date']

In [ ]:
economic_policy_debt = [
    'central_goverment_debt%',
    'tax_revenue%',
    'expense%',
    'inflation_annual%',
    'real_interest_rate'
]

In [ ]:
education = [
    'government_expenditure_on_education%',
    'Literacy rate, adult total (% of people ages 15 and above)',
    'Literacy rate, youth (ages 15-24), gender parity index (GPI)',
    'School enrollment, primary (% net)',
    'School enrollment, secondary (% net)',
    'School enrollment, tertiary (gross), gender parity index (GPI)'
]

In [ ]:
environment = [
    'agricultural_land%',
    'forest_land%',
    'avg_precipitation',
    'co2_emisions',
    'other_greenhouse_emisions',
    'renewvable_energy_consumption%'
]

In [ ]:
financial_sector = [
    'gini_index',
     'gdp_current_us',
]

In [ ]:
gender = [
    'literacy rate, youth (ages 15-24), gender parity index (gpi)',
    'school enrollment, tertiary (gross), gender parity index (gpi)'
]

In [ ]:
health = [
    'life_expectancy_at_birth',
    'birth_rate',
    'death_rate',
    'Mortality rate, infant (per 1,000 live births)',
    'Mortality rate, under-5 (per 1,000 live births)',
    'Government_health_expenditure%'
]

In [ ]:
infrastructure = [
    'access_to_electricity%',
    'electric_power_consumption',
    'individuals_using_internet%',
    'People using at least basic drinking water services (% of population)',
    'People using at least basic sanitation services (% of population)'
]

In [ ]:
poverty = [
    'multidimensional_poverty_headcount_ratio%',
    'unemployment with basic education (% of total labor force with basic education)'
]

In [ ]:
private_sector_trade = [
    'trade_in_services%',
    'Exports of goods and services (% of gdp)',
    'Imports of goods and services (% of gdp)',
    'research_and_development_expenditure%'
]

In [ ]:
public_sector = [
    'control_of_corruption_estimate',
    'control_of_corruption_std',
    'goverment_effectiveness_estimate',
    'goverment_effectiveness_std',
    'political_stability_estimate',
    'political_stability_std',
    'rule_of_law_estimate',
    'rule_of_law_std',
    'regulatory_quality_estimate',
    'regulatory_quality_std',
    'voice_and_accountability_estimate',
    'voice_and_accountability_std'
]

In [ ]:
social_protection_labor = [
    'Labor force participation rate for ages 15-24, total (%) (national estimate)',
    'rural_population'
]

In [ ]:
core_economy = [
    'gdp_current_us',
    'population',
    'population_density'
]

In [ ]:
special = [
    'intentional_homicides',
    'homicide_available'
]

In [ ]:
all_normalized_columns = ['info','economic_policy_debt','education','environment','financial_sector','gender','health','infrastructure','poverty','private_sector_trade','public_sector','social_protection_labor','core_economy','special']

In [ ]:
print(df.columns.tolist())

['Unnamed: 0', 'date', 'agricultural_land%', 'forest_land%', 'land_area', 'avg_precipitation', 'trade_in_services%', 'control_of_corruption_estimate', 'control_of_corruption_std', 'access_to_electricity%', 'renewvable_energy_consumption%', 'electric_power_consumption', 'co2_emisions', 'other_greenhouse_emisions', 'inflation_annual%', 'real_interest_rate', 'research_and_development_expenditure%', 'central_goverment_debt%', 'tax_revenue%', 'expense%', 'goverment_effectiveness_estimate', 'goverment_effectiveness_std', 'individuals_using_internet%', 'military_expenditure%', 'gdp_current_us', 'political_stability_estimate', 'political_stability_std', 'rule_of_law_estimate', 'rule_of_law_std', 'regulatory_quality_estimate', 'regulatory_quality_std', 'government_expenditure_on_education%', 'government_health_expenditure%', 'multidimensional_poverty_headcount_ratio%', 'gini_index', 'birth_rate', 'death_rate', 'life_expectancy_at_birth', 'population', 'rural_population', 'voice_and_accountabili

In [ ]:
print(df.isna().sum().sort_values(ascending=False).head(10))

multidimensional_poverty_headcount_ratio%                                          16490
central_goverment_debt%                                                            13899
unemployment with basic education (% of total labor force with basic education)    13684
research_and_development_expenditure%                                              12999
gini_index                                                                         12908
real_interest_rate                                                                 12674
rule_of_law_estimate                                                               12399
intentional_homicides                                                              12295
goverment_effectiveness_estimate                                                   11765
goverment_effectiveness_std                                                        11765
dtype: int64


In [ ]:
df = pd.read_csv('pre-processed dataset (1).csv')
print(df.columns.tolist())

['Unnamed: 0', 'country', 'date', 'agricultural_land%', 'forest_land%', 'land_area', 'avg_precipitation', 'trade_in_services%', 'control_of_corruption_estimate', 'control_of_corruption_std', 'access_to_electricity%', 'renewvable_energy_consumption%', 'electric_power_consumption', 'co2_emisions', 'other_greenhouse_emisions', 'inflation_annual%', 'real_interest_rate', 'research_and_development_expenditure%', 'central_goverment_debt%', 'tax_revenue%', 'expense%', 'goverment_effectiveness_estimate', 'goverment_effectiveness_std', 'individuals_using_internet%', 'military_expenditure%', 'gdp_current_us', 'political_stability_estimate', 'political_stability_std', 'rule_of_law_estimate', 'rule_of_law_std', 'regulatory_quality_estimate', 'regulatory_quality_std', 'government_expenditure_on_education%', 'government_health_expenditure%', 'multidimensional_poverty_headcount_ratio%', 'gini_index', 'birth_rate', 'death_rate', 'life_expectancy_at_birth', 'population', 'rural_population', 'voice_and_a

In [ ]:
df = df.groupby('country').apply(lambda x: x.fillna(x.mean()))

TypeError: Could not convert ['1960-01-011961-01-011962-01-011963-01-011964-01-011965-01-011966-01-011967-01-011968-01-011969-01-011970-01-011971-01-011972-01-011973-01-011974-01-011975-01-011976-01-011977-01-011978-01-011979-01-011980-01-011981-01-011982-01-011983-01-011984-01-011985-01-011986-01-011987-01-011988-01-011989-01-011990-01-011991-01-011992-01-011993-01-011994-01-011995-01-011996-01-011997-01-011998-01-011999-01-012000-01-012001-01-012002-01-012003-01-012004-01-012005-01-012006-01-012007-01-012008-01-012009-01-012010-01-012011-01-012012-01-012013-01-012014-01-012015-01-012016-01-012017-01-012018-01-012019-01-012020-01-012021-01-012022-01-012023-01-01'] to numeric

In [ ]:
df = df.fillna(0)

In [ ]:
def get_pca_weight(data, cols):
    cols = [c for c in cols if c in data.columns]

    subset = data[cols].copy()
    subset = subset.fillna(subset.mean())

    pca = PCA(n_components=1)
    pca.fit(subset)

    return pca.explained_variance_ratio_[0]  # THIS is your weight

In [ ]:
weights = {}

weights['economic_policy_debt'] = get_pca_weight(df, economic_policy_debt)
weights['education'] = get_pca_weight(df, education)
weights['environment'] = get_pca_weight(df, environment)
weights['financial_sector'] = get_pca_weight(df, financial_sector)


In [ ]:
weights['gender'] = get_pca_weight(df, gender)
weights['health'] = get_pca_weight(df, health)
weights['infrastructure'] = get_pca_weight(df, infrastructure)
weights['poverty'] = get_pca_weight(df, poverty)
weights['private_sector_trade'] = get_pca_weight(df, private_sector_trade)
weights['public_sector'] = get_pca_weight(df, public_sector)
weights['social_protection_labor'] = get_pca_weight(df, social_protection_labor)
weights['core_economy'] = get_pca_weight(df, core_economy)
weights['special'] = get_pca_weight(df, special)

In [ ]:
weights_df = pd.DataFrame(list(weights.items()), columns=['category', 'weight'])

# Normalize weights (VERY IMPORTANT)
weights_df['normalized_weight'] = weights_df['weight'] / weights_df['weight'].sum()

# Rank
weights_df = weights_df.sort_values(by='normalized_weight', ascending=False)

print(weights_df)

               category    weight  normalized_weight
1             education  1.000000           0.269952
2           environment  0.999999           0.269952
0  economic_policy_debt  0.995247           0.268669
3      financial_sector  0.709118           0.191428


In [ ]:
print(df.columns.tolist())

['date', 'agricultural_land%', 'forest_land%', 'land_area', 'avg_precipitation', 'trade_in_services%', 'control_of_corruption_estimate', 'control_of_corruption_std', 'access_to_electricity%', 'renewvable_energy_consumption%', 'electric_power_consumption', 'co2_emisions', 'other_greenhouse_emisions', 'inflation_annual%', 'real_interest_rate', 'research_and_development_expenditure%', 'central_goverment_debt%', 'tax_revenue%', 'expense%', 'goverment_effectiveness_estimate', 'goverment_effectiveness_std', 'individuals_using_internet%', 'military_expenditure%', 'gdp_current_us', 'political_stability_estimate', 'political_stability_std', 'rule_of_law_estimate', 'rule_of_law_std', 'regulatory_quality_estimate', 'regulatory_quality_std', 'government_expenditure_on_education%', 'government_health_expenditure%', 'multidimensional_poverty_headcount_ratio%', 'gini_index', 'birth_rate', 'death_rate', 'life_expectancy_at_birth', 'population', 'rural_population', 'voice_and_accountability_estimate', 

In [ ]:
df_scores = df[['country', 'date']].copy()

df_scores['economic_policy_debt'] = get_pca_score(df, economic_policy_debt)
df_scores['education'] = get_pca_score(df, education)
df_scores['environment'] = get_pca_score(df, environment)
df_scores['financial_sector'] = get_pca_score(df, financial_sector)
df_scores['gender'] = get_pca_score(df, gender)
df_scores['health'] = get_pca_score(df, health)
df_scores['infrastructure'] = get_pca_score(df, infrastructure)
df_scores['poverty'] = get_pca_score(df, poverty)
df_scores['private_sector_trade'] = get_pca_score(df, private_sector_trade)
df_scores['public_sector'] = get_pca_score(df, public_sector)
df_scores['social_protection_labor'] = get_pca_score(df, social_protection_labor)
df_scores['core_economy'] = get_pca_score(df, core_economy)
df_scores['special'] = get_pca_score(df, special)

KeyError: "['country'] not in index"